# Random Forest com Scikit-Learn e Vega Datasets: Stocks

**Atividade de Inteligência Artificial e Aprendizagem de Máquina — Ciência da Computação**

**Autores:** Nicolas Mariano da Silva ([`@nickmds1`](https://github.com/nickmds1), RA `0033311`) e Pedro Henrique Isamu Fagundes de Souza Tsukahara Yoshissaro ([`@knutzin`](https://github.com/knutzin), RA `00330820`).

Neste laboratório, usamos o conjunto **Stocks** do `vega_datasets` para explorar uma tarefa de **regressão** com `RandomForestRegressor`. A meta didática é estimar o preço mensal de cada ticker com base na data e em observações passadas. O conjunto possui 560 registros mensais de cinco empresas entre 2000 e 2010.

> É um exercício de aprendizado de máquina com dados históricos; não é recomendação nem ferramenta de investimento.


## 1. Instalação e imports

No Colab ou Jupyter, a instalação pode ser feita executando `%pip install -r requirements.txt` no diretório do projeto. A célula abaixo instala somente o pacote do Vega se ele ainda não estiver disponível.


In [ ]:
%pip -q install vega_datasets


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from vega_datasets import data
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


## 2. Carregar e conhecer os dados

O dataset tem três colunas: ticker (`symbol`), data mensal (`date`) e preço (`price`). Vamos converter a data, conferir os registros e verificar se há valores ausentes.


In [ ]:
stocks = data.stocks().copy()
stocks["date"] = pd.to_datetime(stocks["date"])
stocks = stocks.sort_values(["symbol", "date"]).reset_index(drop=True)

print(f"Dimensões: {stocks.shape[0]} linhas × {stocks.shape[1]} colunas")
print(f"Período: {stocks['date'].min():%Y-%m} a {stocks['date'].max():%Y-%m}")
print(f"Tickers: {', '.join(sorted(stocks['symbol'].unique()))}")
display(stocks.head())
print("\nValores ausentes por coluna:")
display(stocks.isna().sum().to_frame("valores ausentes"))


## 3. Exploração visual

As séries têm escalas e trajetórias diferentes. O gráfico ajuda a observar essa variação antes de preparar os atributos para o modelo.


In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
for symbol, grupo in stocks.groupby("symbol"):
    ax.plot(grupo["date"], grupo["price"], label=symbol, linewidth=1.8)
ax.set(title="Preços mensais no conjunto Vega Stocks (2000–2010)", xlabel="Data", ylabel="Preço")
ax.legend(title="Ticker", ncol=2)
ax.grid(alpha=0.25)
fig.tight_layout()
plt.show()


## 4. Preparar atributos sem antecipar o futuro

Para cada ticker, criamos atributos usando apenas observações anteriores: preço do mês anterior (`lag_1`), preço de três meses antes (`lag_3`) e média dos três meses anteriores. Também codificamos a data e o ticker.

As primeiras linhas de cada série não têm histórico suficiente para esses atributos e são removidas. A separação de treino e teste será feita por data: os meses mais antigos treinam o modelo e os mais recentes ficam reservados para avaliação.


In [ ]:
model_data = stocks.copy()
groups = model_data.groupby("symbol")["price"]
model_data["lag_1"] = groups.shift(1)
model_data["lag_3"] = groups.shift(3)
model_data["media_movel_3"] = groups.transform(lambda s: s.shift(1).rolling(3).mean())
model_data["ano"] = model_data["date"].dt.year
model_data["mes"] = model_data["date"].dt.month
model_data = model_data.dropna().reset_index(drop=True)

cutoff = sorted(model_data["date"].unique())[int(model_data["date"].nunique() * 0.8)]
train = model_data[model_data["date"] < cutoff].copy()
test = model_data[model_data["date"] >= cutoff].copy()

print(f"Data de corte: {pd.Timestamp(cutoff):%Y-%m}")
print(f"Treino: {len(train)} linhas | Teste: {len(test)} linhas")
print(f"Período de treino: {train['date'].min():%Y-%m} a {train['date'].max():%Y-%m}")
print(f"Período de teste: {test['date'].min():%Y-%m} a {test['date'].max():%Y-%m}")


## 5. Treinar o Random Forest

O ticker é transformado em colunas indicadoras; as demais variáveis são numéricas. A divisão temporal impede que meses mais recentes sejam usados para treinar a avaliação de meses anteriores.


In [ ]:
feature_cols = ["symbol", "ano", "mes", "lag_1", "lag_3", "media_movel_3"]
X_train = pd.get_dummies(train[feature_cols], columns=["symbol"], dtype=int)
X_test = pd.get_dummies(test[feature_cols], columns=["symbol"], dtype=int)
X_test = X_test.reindex(columns=X_train.columns, fill_value=0)
y_train = train["price"]
y_test = test["price"]

rf = RandomForestRegressor(
    n_estimators=300,
    min_samples_leaf=2,
    random_state=42,
    n_jobs=-1,
)
rf.fit(X_train, y_train)
y_pred = rf.predict(X_test)


## 6. Avaliar e comparar com uma referência simples

Além do Random Forest, usamos como referência a previsão ingênua que repete o último preço observado (`lag_1`). O MAE e o RMSE medem o tamanho dos erros, na unidade de preço; o R² compara o modelo a uma referência baseada na média do alvo. O R² pode ser negativo quando as previsões são piores que essa referência.


In [ ]:
def avaliar(nome, y_real, y_estimado):
    return {
        "Modelo": nome,
        "MAE": mean_absolute_error(y_real, y_estimado),
        "RMSE": np.sqrt(mean_squared_error(y_real, y_estimado)),
        "R²": r2_score(y_real, y_estimado),
    }

resultados = pd.DataFrame([
    avaliar("Random Forest", y_test, y_pred),
    avaliar("Último preço (lag_1)", y_test, test["lag_1"]),
]).set_index("Modelo")
display(resultados.round(3))


## 7. Observar previsões no período de teste

O gráfico compara valores observados e estimados ao longo do período reservado. A previsão usa defasagens observadas mês a mês, representando um cenário de previsão de um passo à frente.


In [ ]:
predicoes = test[["symbol", "date", "price", "lag_1"]].copy()
predicoes["preco_previsto"] = y_pred

fig, axes = plt.subplots(3, 2, figsize=(14, 10), sharex=True)
for ax, (symbol, grupo) in zip(axes.flat, predicoes.groupby("symbol")):
    ax.plot(grupo["date"], grupo["price"], marker="o", label="Observado")
    ax.plot(grupo["date"], grupo["preco_previsto"], marker="x", linestyle="--", label="Random Forest")
    ax.set_title(symbol)
    ax.set_ylabel("Preço")
    ax.grid(alpha=0.25)
    ax.legend()
for ax in axes.flat[len(predicoes["symbol"].unique()):]:
    ax.set_visible(False)
fig.suptitle("Preços observados e previstos — período de teste", y=1.01)
fig.tight_layout()
plt.show()


## 8. Importância das variáveis

A importância mostra como as variáveis foram usadas pelas árvores deste modelo. Ela é descritiva, depende dos dados e da configuração escolhida e não indica causalidade.


In [ ]:
importancias = pd.Series(rf.feature_importances_, index=X_train.columns).sort_values()
ax = importancias.plot(kind="barh", figsize=(9, 5), color="teal", title="Importância das variáveis — Random Forest")
ax.set_xlabel("Importância")
ax.set_ylabel("Atributo")
ax.grid(axis="x", alpha=0.25)
plt.tight_layout()
plt.show()
display(importancias.sort_values(ascending=False).rename("importância").to_frame().round(3))


## 9. Conclusão

Ao executar o notebook, registre:

1. Qual foi o MAE do Random Forest e como ele se compara à referência `lag_1`?
2. O desempenho variou entre os tickers? Observe os gráficos por empresa.
3. Quais atributos tiveram maior importância?
4. Quais limitações aparecem por usar apenas 560 observações mensais antigas?

O exercício demonstra uma aplicação de regressão com Random Forest e uma avaliação temporal simples. Ele não modela todos os fatores que afetam preços de ativos e não deve ser usado para tomar decisões financeiras.

## Referências

- [Vega Datasets — Stocks](https://vega.github.io/vega-datasets/datasets/stocks/)
- [Scikit-Learn — RandomForestRegressor](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestRegressor.html)
- [Scikit-Learn — métricas de regressão](https://scikit-learn.org/stable/modules/model_evaluation.html#regression-metrics)
